# Scaling Laws

A common question in neural network training is, how should I select my hyperparameters? While a proper hyperparameter sweep will always provide the best answer, sweeps can become impractical especially at larger network sizes. In this case, the field has converged to two main options: 1), copy what a previous project used (which is always a good starting point), or 2) derive *scaling laws* which can predict what the best hyperparameters will be.

In this homework question, we will derive a simple scaling law for the optimal learning rate under varying batch sizes.

## Setup and Configuration

First, we organize imports following PEP 8 guidelines:
1. Standard library imports
2. Third-party imports
3. Local imports

We also define configuration constants using UPPER_SNAKE_CASE naming convention.

In [ ]:
"""Scaling Laws for Learning Rate and Batch Size.

This notebook explores the relationship between optimal learning rate
and batch size for various optimization algorithms.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Callable, Literal

import matplotlib.pyplot as plt
import numpy as np
import torch
from numpy.typing import NDArray

In [ ]:
# =============================================================================
# Configuration Constants (UPPER_SNAKE_CASE per PEP 8)
# =============================================================================

# Reproducibility seed
RANDOM_SEED = 0

# Linear regression dataset configuration
LINEAR_NUM_SAMPLES = 10_000
LINEAR_INPUT_DIM = 16
LINEAR_NOISE_STD = 5
LINEAR_TEST_SAMPLES = 1_000

# MLP dataset configuration
MLP_NUM_SAMPLES = 10_000
MLP_INPUT_DIM = 4
MLP_NOISE_STD = 2
MLP_TEST_SAMPLES = 1_000

# Training configuration
DEFAULT_ITERATIONS = 100
DEFAULT_BATCH_SIZE = 32
DEFAULT_LEARNING_RATE = 0.01

# Learning rate sweep configuration
BATCH_SIZES = [2, 4, 16, 64, 256, 1024, 2048]
LR_BASE = 0.001
LR_MULTIPLIER = 1.25
NUM_LR_STEPS = 32

# Visualization configuration
FIGURE_SIZE = (10, 6)
FIGURE_SIZE_LARGE = (12, 8)
LOSS_CLIP_VALUE = 500
DIVERGENCE_THRESHOLD = 1e6
OPTIMAL_LR_THRESHOLD = 1.05  # 5% above minimum

## Data Classes for Structured Data

Using Python dataclasses provides:
- Type safety and documentation
- Immutable data structures (frozen=True)
- Clear interface between functions

In [ ]:
@dataclass(frozen=True)
class LinearDataset:
    """Container for linear regression dataset.
    
    Attributes:
        x_train: Training input features of shape (num_samples, input_dim).
        y_train: Training targets of shape (num_samples, output_dim).
        x_test: Test input features of shape (test_samples, input_dim).
        y_test: Test targets of shape (test_samples, output_dim).
        w_true: Ground truth weight matrix of shape (input_dim, output_dim).
        noise_std: Standard deviation of additive Gaussian noise.
    """
    x_train: NDArray[np.floating]
    y_train: NDArray[np.floating]
    x_test: NDArray[np.floating]
    y_test: NDArray[np.floating]
    w_true: NDArray[np.floating]
    noise_std: float


@dataclass(frozen=True)
class MLPDataset:
    """Container for MLP regression dataset.
    
    Attributes:
        x_train: Training input features (NumPy array).
        y_train: Training targets (NumPy array).
        x_test: Test input features (PyTorch tensor).
        y_test: Test targets (PyTorch tensor).
        noise_std: Standard deviation of additive Gaussian noise.
        input_dim: Dimension of input features.
    """
    x_train: NDArray[np.floating]
    y_train: NDArray[np.floating]
    x_test: torch.Tensor
    y_test: torch.Tensor
    noise_std: float
    input_dim: int


@dataclass
class TrainingResult:
    """Container for training results.
    
    Attributes:
        train_losses: List of training losses per iteration.
        test_losses: List of test losses per iteration.
        final_weights: Final model weights (optional, for linear regression).
    """
    train_losses: list[float]
    test_losses: list[float]
    final_weights: NDArray[np.floating] | None = None


@dataclass
class ScalingLawResult:
    """Container for scaling law analysis results.
    
    Attributes:
        batch_sizes: List of batch sizes tested.
        optimal_lrs: Optimal learning rate for each batch size.
        avg_optimal_lrs: Average LR within threshold of optimal.
        power_law_slope: Fitted power law exponent.
        results: Detailed results dictionary {batch_size: {lr: loss}}.
    """
    batch_sizes: list[int]
    optimal_lrs: list[float]
    avg_optimal_lrs: list[float]
    power_law_slope: float
    results: dict[int, dict[float, float]]

## Dataset Generation Functions

These functions generate synthetic datasets for our experiments. Key improvements:
- Type hints for all parameters and return values
- Google-style docstrings with Args, Returns sections
- Explicit random state management for reproducibility

In [ ]:
def create_linear_dataset(
    num_samples: int = LINEAR_NUM_SAMPLES,
    test_samples: int = LINEAR_TEST_SAMPLES,
    input_dim: int = LINEAR_INPUT_DIM,
    noise_std: float = LINEAR_NOISE_STD,
    seed: int = RANDOM_SEED,
) -> LinearDataset:
    """Generate synthetic linear regression dataset.
    
    Creates a dataset where y = X @ w_true + noise, with Gaussian noise.
    
    Args:
        num_samples: Number of training samples to generate.
        test_samples: Number of test samples to generate.
        input_dim: Dimension of input features.
        noise_std: Standard deviation of additive Gaussian noise.
        seed: Random seed for reproducibility.
    
    Returns:
        LinearDataset containing train/test data and ground truth.
    """
    rng = np.random.default_rng(seed)
    
    x_train = rng.standard_normal((num_samples, input_dim))
    x_test = rng.standard_normal((test_samples, input_dim))
    w_true = rng.standard_normal((input_dim, input_dim))
    
    y_train = x_train @ w_true + noise_std * rng.standard_normal((num_samples, input_dim))
    y_test = x_test @ w_true + noise_std * rng.standard_normal((test_samples, input_dim))
    
    return LinearDataset(
        x_train=x_train,
        y_train=y_train,
        x_test=x_test,
        y_test=y_test,
        w_true=w_true,
        noise_std=noise_std,
    )


def create_mlp_dataset(
    num_samples: int = MLP_NUM_SAMPLES,
    test_samples: int = MLP_TEST_SAMPLES,
    input_dim: int = MLP_INPUT_DIM,
    noise_std: float = MLP_NOISE_STD,
    seed: int = RANDOM_SEED,
) -> MLPDataset:
    """Generate synthetic MLP regression dataset.
    
    Creates a dataset where y = relu(X @ W1) @ W2 + noise.
    
    Args:
        num_samples: Number of training samples to generate.
        test_samples: Number of test samples to generate.
        input_dim: Dimension of input features.
        noise_std: Standard deviation of additive Gaussian noise.
        seed: Random seed for reproducibility.
    
    Returns:
        MLPDataset containing train/test data with test data as tensors.
    """
    rng = np.random.default_rng(seed)
    
    x_train = rng.standard_normal((num_samples, input_dim))
    x_test = rng.standard_normal((test_samples, input_dim))
    w1_true = rng.standard_normal((input_dim, input_dim))
    w2_true = rng.standard_normal((input_dim, input_dim))
    
    y_train = np.maximum(0, x_train @ w1_true) @ w2_true + noise_std * rng.standard_normal((num_samples, input_dim))
    y_test = np.maximum(0, x_test @ w1_true) @ w2_true + noise_std * rng.standard_normal((test_samples, input_dim))
    
    return MLPDataset(
        x_train=x_train,
        y_train=y_train,
        x_test=torch.tensor(x_test, dtype=torch.float32),
        y_test=torch.tensor(y_test, dtype=torch.float32),
        noise_std=noise_std,
        input_dim=input_dim,
    )

## Linear Regression Training

First, let's consider the case of a simple least-squares gradient descent problem. We will define our dataset using a randomly-sampled ground truth linear mapping, and our training samples will be augmented by some amount of noise. For this homework, we will focus on the question of **how should learning rate scale with batch size?**

In [ ]:
def compute_mse(
    x: NDArray[np.floating],
    y: NDArray[np.floating],
    weights: NDArray[np.floating],
) -> float:
    """Compute mean squared error for linear regression.
    
    Args:
        x: Input features of shape (num_samples, input_dim).
        y: Target values of shape (num_samples, output_dim).
        weights: Weight matrix of shape (input_dim, output_dim).
    
    Returns:
        Mean squared error as a scalar float.
    """
    predictions = x @ weights
    return float(np.mean((y - predictions) ** 2))


def compute_analytical_solution(
    x: NDArray[np.floating],
    y: NDArray[np.floating],
    num_samples: int,
) -> NDArray[np.floating]:
    """Compute analytical least-squares solution using pseudo-inverse.
    
    Args:
        x: Input features of shape (total_samples, input_dim).
        y: Target values of shape (total_samples, output_dim).
        num_samples: Number of samples to use from the beginning.
    
    Returns:
        Optimal weight matrix of shape (input_dim, output_dim).
    """
    x_subset = x[:num_samples, :]
    y_subset = y[:num_samples, :]
    pseudo_inverse = np.linalg.pinv(x_subset, rcond=0)
    return pseudo_inverse @ y_subset


def train_linear_sgd(
    dataset: LinearDataset,
    num_iterations: int = DEFAULT_ITERATIONS,
    batch_size: int = DEFAULT_BATCH_SIZE,
    learning_rate: float = DEFAULT_LEARNING_RATE,
    seed: int = RANDOM_SEED,
) -> TrainingResult:
    """Train linear regression using stochastic gradient descent.
    
    Performs mini-batch SGD on the least-squares objective:
    L(w) = (1/n) * ||Xw - y||^2
    
    Args:
        dataset: LinearDataset containing train and test data.
        num_iterations: Number of SGD iterations to perform.
        batch_size: Number of samples per mini-batch.
        learning_rate: Step size for gradient descent.
        seed: Random seed for reproducibility.
    
    Returns:
        TrainingResult containing loss histories and final weights.
    """
    rng = np.random.default_rng(seed)
    num_samples = dataset.x_train.shape[0]
    input_dim = dataset.x_train.shape[1]
    
    # Initialize weights randomly
    weights = rng.standard_normal((input_dim, input_dim))
    
    train_losses: list[float] = []
    test_losses: list[float] = []
    
    for _ in range(num_iterations):
        # Sample mini-batch indices
        batch_indices = rng.integers(0, num_samples, size=batch_size)
        x_batch = dataset.x_train[batch_indices]
        y_batch = dataset.y_train[batch_indices]
        
        # Compute gradient: d/dw [(1/n) * ||Xw - y||^2] = (2/n) * X^T(Xw - y)
        gradient = (2 / batch_size) * (x_batch.T @ x_batch @ weights - x_batch.T @ y_batch)
        
        # Update weights
        weights = weights - learning_rate * gradient
        
        # Record losses
        train_losses.append(compute_mse(x_batch, y_batch, weights))
        test_losses.append(compute_mse(dataset.x_test, dataset.y_test, weights))
    
    return TrainingResult(
        train_losses=train_losses,
        test_losses=test_losses,
        final_weights=weights,
    )

## Visualization Utilities

Dedicated plotting functions for reusability and consistent styling.

In [ ]:
def plot_training_curves(
    result: TrainingResult,
    title: str = "Training Progress",
    figsize: tuple[int, int] = FIGURE_SIZE,
) -> None:
    """Plot train and test loss curves.
    
    Args:
        result: TrainingResult containing loss histories.
        title: Title for the plot.
        figsize: Figure dimensions (width, height).
    """
    plt.figure(figsize=figsize)
    plt.plot(result.train_losses, label="Train Loss")
    plt.plot(result.test_losses, label="Test Loss")
    plt.xlabel("Number of Training Iterations")
    plt.ylabel("Squared Error")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_residual_curves(
    result: TrainingResult,
    irreducible_error: float,
    title: str = "Convergence Analysis",
    figsize: tuple[int, int] = FIGURE_SIZE,
) -> None:
    """Plot log-scale residual loss curves (loss - irreducible error).
    
    Args:
        result: TrainingResult containing loss histories.
        irreducible_error: Baseline error to subtract (typically sigma^2).
        title: Title for the plot.
        figsize: Figure dimensions (width, height).
    """
    train_residuals = [loss - irreducible_error for loss in result.train_losses]
    test_residuals = [loss - irreducible_error for loss in result.test_losses]
    
    plt.figure(figsize=figsize)
    plt.plot(train_residuals, label="Train Loss")
    plt.plot(test_residuals, label="Test Loss")
    plt.yscale("log")
    plt.xlabel("Number of Training Iterations")
    plt.ylabel(r"Squared Error - $\sigma^2$")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_lr_sweep_results(
    results: dict[int, dict[float, float]],
    batch_sizes: list[int],
    title: str,
    clip_value: float = LOSS_CLIP_VALUE,
    figsize: tuple[int, int] = FIGURE_SIZE_LARGE,
) -> None:
    """Plot learning rate vs loss for multiple batch sizes.
    
    Args:
        results: Nested dict {batch_size: {learning_rate: final_loss}}.
        batch_sizes: List of batch sizes to plot.
        title: Title for the plot.
        clip_value: Maximum value to clip losses for visualization.
        figsize: Figure dimensions (width, height).
    """
    plt.figure(figsize=figsize)
    
    for batch_size in batch_sizes:
        learning_rates = list(results[batch_size].keys())
        losses = [results[batch_size][lr] for lr in learning_rates]
        losses_clipped = np.clip(losses, 0, clip_value)
        plt.plot(
            learning_rates,
            losses_clipped,
            label=f"Batch Size {batch_size}",
            marker="o",
            markersize=3,
        )
    
    plt.xscale("log")
    plt.xlabel("Learning Rate")
    plt.ylabel("Test Loss (clipped)")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()


def plot_scaling_law(
    scaling_result: ScalingLawResult,
    title: str,
    figsize: tuple[int, int] = FIGURE_SIZE,
) -> None:
    """Plot batch size vs optimal learning rate with power law fit.
    
    Args:
        scaling_result: ScalingLawResult containing analysis results.
        title: Title for the plot.
        figsize: Figure dimensions (width, height).
    """
    plt.figure(figsize=figsize)
    
    # Plot optimal and average LRs
    plt.plot(
        scaling_result.batch_sizes,
        scaling_result.optimal_lrs,
        "bo-",
        label="Optimal LR",
        markersize=8,
    )
    plt.plot(
        scaling_result.batch_sizes,
        scaling_result.avg_optimal_lrs,
        "rs--",
        label="Avg LR (within 5% of optimal)",
        markersize=8,
    )
    
    # Plot power law fit
    log_batch_sizes = np.log(scaling_result.batch_sizes)
    log_lrs = np.log(scaling_result.avg_optimal_lrs)
    _, intercept = np.polyfit(log_batch_sizes, log_lrs, 1)
    fitted_lrs = np.exp(intercept + scaling_result.power_law_slope * log_batch_sizes)
    
    plt.plot(
        scaling_result.batch_sizes,
        fitted_lrs,
        "g--",
        linewidth=2,
        label=f"Power law fit: LR ~ BS^{scaling_result.power_law_slope:.2f}",
    )
    
    plt.xscale("log")
    plt.yscale("log")
    plt.xlabel("Batch Size")
    plt.ylabel("Optimal Learning Rate")
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

## Demo: Linear Regression Training

Let's demonstrate the basic training loop before the scaling law experiments.

In [ ]:
# Create dataset
linear_dataset = create_linear_dataset()

# Train with SGD
linear_result = train_linear_sgd(
    dataset=linear_dataset,
    num_iterations=DEFAULT_ITERATIONS,
    batch_size=DEFAULT_BATCH_SIZE,
    learning_rate=DEFAULT_LEARNING_RATE,
)

# Plot results
plot_training_curves(linear_result, title="Linear Regression SGD Training")

We can plot the above curves on log-linear scale while subtracting off the irreducible error of $\sigma^2$ to see the linear decay of the squared error. This allows for a better view of the convergence rate.

In [ ]:
irreducible_error = linear_dataset.noise_std ** 2
plot_residual_curves(
    linear_result,
    irreducible_error=irreducible_error,
    title="Linear Regression Convergence (Log Scale)",
)

## Q1: Scaling Law for Least-Squares SGD

Perform a sweep over learning rates. Consider the batch sizes between [2, 4, 16, 64, 256, 1024, 2048], and sweep over learning rates logarithmically with a resolution of $1.25$. For example, you should consider the learning rate of $0.001, 0.001 \times 1.25, 0.001 \times 1.25^2$, etc. **Make sure your learning rate sweep covers the optimal LR for each batch size (e.g. your optimal LR should not be at the boundary of your learning rate intervals.)** You should be able to sweep around ~32 learning rates per batch size.

- Plot your learning rates on the same graph, with each batchsize as a different curve. Your curve should resemble the example provided.
- Plot the relationship between batch size (x-axis) and the optimal learning rate (y-axis). What function does this resemble?

**Hint:** Many runs will result in very high losses if the iteration diverges. It will help to clip the losses to some ceiling value before plotting.

**Hint 2:** You may find that some batch sizes have a wide basin of optimal learning rates that perform roughly equivalently. In this case, it may help to plot the *average* learning rate that is within X% of the optimal. This can make the relationship more clear.

![fig-example](https://github.com/Berkeley-CS182/cs182fa25_public/blob/main/hw11/code/fig-example.png?raw=1)

In [ ]:
def generate_learning_rates(
    base: float = LR_BASE,
    multiplier: float = LR_MULTIPLIER,
    num_steps: int = NUM_LR_STEPS,
) -> list[float]:
    """Generate logarithmically-spaced learning rates.
    
    Args:
        base: Starting learning rate.
        multiplier: Factor to multiply by for each step.
        num_steps: Total number of learning rates to generate.
    
    Returns:
        List of learning rates in ascending order.
    """
    return [base * (multiplier ** i) for i in range(num_steps)]


def compute_final_loss(
    test_losses: list[float],
    num_final_iters: int = 10,
    divergence_threshold: float = DIVERGENCE_THRESHOLD,
) -> float:
    """Compute final loss as mean of last iterations, with divergence handling.
    
    Args:
        test_losses: List of test losses per iteration.
        num_final_iters: Number of final iterations to average.
        divergence_threshold: Maximum allowed loss before clamping.
    
    Returns:
        Final loss value, clamped if diverged.
    """
    final_loss = np.mean(test_losses[-num_final_iters:])
    
    if np.isnan(final_loss) or np.isinf(final_loss):
        return divergence_threshold
    
    return min(float(final_loss), divergence_threshold)


def find_optimal_learning_rates(
    results: dict[int, dict[float, float]],
    batch_sizes: list[int],
    threshold_factor: float = OPTIMAL_LR_THRESHOLD,
) -> tuple[list[float], list[float]]:
    """Find optimal and average-optimal learning rates per batch size.
    
    Args:
        results: Nested dict {batch_size: {learning_rate: final_loss}}.
        batch_sizes: List of batch sizes to analyze.
        threshold_factor: Factor above minimum to include in average.
    
    Returns:
        Tuple of (optimal_lrs, avg_optimal_lrs) lists.
    """
    optimal_lrs: list[float] = []
    avg_optimal_lrs: list[float] = []
    
    for batch_size in batch_sizes:
        learning_rates = list(results[batch_size].keys())
        losses = [results[batch_size][lr] for lr in learning_rates]
        
        # Find minimum loss and corresponding LR
        min_loss = min(losses)
        optimal_idx = losses.index(min_loss)
        optimal_lr = learning_rates[optimal_idx]
        optimal_lrs.append(optimal_lr)
        
        # Find geometric mean of LRs within threshold of optimal
        threshold = min_loss * threshold_factor
        lrs_within_threshold = [
            learning_rates[i]
            for i in range(len(learning_rates))
            if losses[i] <= threshold
        ]
        
        if lrs_within_threshold:
            # Geometric mean is appropriate for log-scaled values
            avg_lr = float(np.exp(np.mean(np.log(lrs_within_threshold))))
        else:
            avg_lr = optimal_lr
        avg_optimal_lrs.append(avg_lr)
    
    return optimal_lrs, avg_optimal_lrs


def fit_power_law(batch_sizes: list[int], learning_rates: list[float]) -> float:
    """Fit power law relationship: lr ~ batch_size^slope.
    
    Args:
        batch_sizes: List of batch sizes.
        learning_rates: Corresponding optimal learning rates.
    
    Returns:
        Power law exponent (slope in log-log space).
    """
    log_batch_sizes = np.log(batch_sizes)
    log_lrs = np.log(learning_rates)
    slope, _ = np.polyfit(log_batch_sizes, log_lrs, 1)
    return float(slope)

In [ ]:
def run_linear_lr_sweep(
    dataset: LinearDataset,
    batch_sizes: list[int] = BATCH_SIZES,
    num_iterations: int = DEFAULT_ITERATIONS,
    verbose: bool = True,
) -> ScalingLawResult:
    """Run learning rate sweep for linear SGD across batch sizes.
    
    Args:
        dataset: LinearDataset to train on.
        batch_sizes: List of batch sizes to test.
        num_iterations: Number of training iterations per run.
        verbose: Whether to print progress.
    
    Returns:
        ScalingLawResult containing sweep results and analysis.
    """
    learning_rates = generate_learning_rates()
    results: dict[int, dict[float, float]] = {}
    
    for batch_size in batch_sizes:
        results[batch_size] = {}
        
        for lr in learning_rates:
            try:
                result = train_linear_sgd(
                    dataset=dataset,
                    num_iterations=num_iterations,
                    batch_size=batch_size,
                    learning_rate=lr,
                )
                final_loss = compute_final_loss(result.test_losses)
            except Exception:
                final_loss = DIVERGENCE_THRESHOLD
            
            results[batch_size][lr] = final_loss
        
        if verbose:
            print(f"Finished batch size {batch_size}")
    
    # Analyze results
    optimal_lrs, avg_optimal_lrs = find_optimal_learning_rates(results, batch_sizes)
    power_law_slope = fit_power_law(batch_sizes, avg_optimal_lrs)
    
    return ScalingLawResult(
        batch_sizes=batch_sizes,
        optimal_lrs=optimal_lrs,
        avg_optimal_lrs=avg_optimal_lrs,
        power_law_slope=power_law_slope,
        results=results,
    )

In [ ]:
#### TODO: Q1 - Run the scaling law experiment for linear SGD

# Run learning rate sweep
linear_scaling_result = run_linear_lr_sweep(linear_dataset)

# Plot LR sweep results
plot_lr_sweep_results(
    linear_scaling_result.results,
    linear_scaling_result.batch_sizes,
    title="Learning Rate vs Test Loss (Least-Squares SGD)",
)

# Plot scaling law
plot_scaling_law(
    linear_scaling_result,
    title="Batch Size vs Optimal Learning Rate (Least-Squares SGD)",
)

print(f"\nThe relationship resembles a power law: LR ~ BS^{linear_scaling_result.power_law_slope:.2f}")
print("This is consistent with the linear scaling rule where LR scales approximately linearly with batch size.")

####

## Q2: MLP with SGD

We will now repeat a similar study, using a two-layer MLP rather than a linear regression. Again, conduct a sweep on the relationship between batch size and optimal learning rate. How does this relationship compare to the optimum for least-squares SGD?

In [ ]:
def create_mlp_model(input_dim: int) -> tuple[torch.nn.Module, torch.nn.Module]:
    """Create a two-layer MLP model with ReLU activation.
    
    Args:
        input_dim: Dimension of input features (also used for hidden and output).
    
    Returns:
        Tuple of (model, loss_function).
    """
    model = torch.nn.Sequential(
        torch.nn.Linear(input_dim, input_dim),
        torch.nn.ReLU(),
        torch.nn.Linear(input_dim, input_dim),
    )
    loss_fn = torch.nn.MSELoss()
    return model, loss_fn


OptimizerType = Literal["SGD", "Adam"]


def create_optimizer_and_scheduler(
    model: torch.nn.Module,
    optimizer_type: OptimizerType,
    learning_rate: float,
    num_iterations: int,
) -> tuple[torch.optim.Optimizer, torch.optim.lr_scheduler.LRScheduler | None]:
    """Create optimizer and optional learning rate scheduler.
    
    Args:
        model: PyTorch model to optimize.
        optimizer_type: Either 'SGD' or 'Adam'.
        learning_rate: Base learning rate.
        num_iterations: Total training iterations (for scheduler).
    
    Returns:
        Tuple of (optimizer, scheduler). Scheduler is None for SGD.
    
    Raises:
        ValueError: If optimizer_type is not recognized.
    """
    if optimizer_type == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
        scheduler = None
    elif optimizer_type == "Adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=learning_rate,
            betas=(0.5, 0.5),
            weight_decay=0.01,
        )
        # Linear warmup for 10 steps, then cosine decay
        warmup_steps = 10
        warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
            optimizer,
            start_factor=0.001,
            end_factor=1.0,
            total_iters=warmup_steps,
        )
        cosine_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=num_iterations - warmup_steps,
        )
        scheduler = torch.optim.lr_scheduler.SequentialLR(
            optimizer,
            schedulers=[warmup_scheduler, cosine_scheduler],
            milestones=[warmup_steps],
        )
    else:
        raise ValueError(f"Unknown optimizer type: {optimizer_type}")
    
    return optimizer, scheduler


def train_mlp(
    dataset: MLPDataset,
    num_iterations: int = DEFAULT_ITERATIONS,
    batch_size: int = DEFAULT_BATCH_SIZE,
    learning_rate: float = DEFAULT_LEARNING_RATE,
    optimizer_type: OptimizerType = "SGD",
    seed: int = RANDOM_SEED,
) -> TrainingResult:
    """Train MLP using specified optimizer.
    
    Args:
        dataset: MLPDataset containing train and test data.
        num_iterations: Number of training iterations.
        batch_size: Number of samples per mini-batch.
        learning_rate: Base learning rate.
        optimizer_type: Either 'SGD' or 'Adam'.
        seed: Random seed for reproducibility.
    
    Returns:
        TrainingResult containing loss histories.
    """
    # Set seeds for reproducibility
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    
    # Create model and optimizer
    model, loss_fn = create_mlp_model(dataset.input_dim)
    optimizer, scheduler = create_optimizer_and_scheduler(
        model, optimizer_type, learning_rate, num_iterations
    )
    
    num_samples = dataset.x_train.shape[0]
    train_losses: list[float] = []
    test_losses: list[float] = []
    
    for _ in range(num_iterations):
        # Sample mini-batch
        batch_indices = rng.integers(0, num_samples, size=batch_size)
        x_batch = torch.tensor(dataset.x_train[batch_indices], dtype=torch.float32)
        y_batch = torch.tensor(dataset.y_train[batch_indices], dtype=torch.float32)
        
        # Forward pass
        predictions = model(x_batch)
        loss = loss_fn(predictions, y_batch)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        if scheduler is not None:
            scheduler.step()
        
        # Record losses
        with torch.no_grad():
            test_loss = loss_fn(model(dataset.x_test), dataset.y_test)
            train_loss = loss_fn(model(x_batch), y_batch)
        
        train_losses.append(train_loss.item())
        test_losses.append(test_loss.item())
    
    return TrainingResult(
        train_losses=train_losses,
        test_losses=test_losses,
    )

In [ ]:
# Create MLP dataset
mlp_dataset = create_mlp_dataset()

# Demo training
mlp_result = train_mlp(
    dataset=mlp_dataset,
    num_iterations=DEFAULT_ITERATIONS,
    batch_size=256,
    learning_rate=0.1,
    optimizer_type="SGD",
)

plot_training_curves(mlp_result, title="MLP SGD Training")

Again, let's plot on log-linear scale minus the offset. Notice that the MLP converges to $\approx \sigma^2 + 0.57$ as opposed to $\sigma^2$.

In [ ]:
# MLP has additional irreducible error beyond noise variance
MLP_IRREDUCIBLE_OFFSET = 0.57
mlp_irreducible_error = mlp_dataset.noise_std ** 2 + MLP_IRREDUCIBLE_OFFSET

plot_residual_curves(
    mlp_result,
    irreducible_error=mlp_irreducible_error,
    title="MLP Convergence (Log Scale)",
)

In [ ]:
def run_mlp_lr_sweep(
    dataset: MLPDataset,
    optimizer_type: OptimizerType = "SGD",
    batch_sizes: list[int] = BATCH_SIZES,
    num_iterations: int = DEFAULT_ITERATIONS,
    lr_base: float | None = None,
    verbose: bool = True,
) -> ScalingLawResult:
    """Run learning rate sweep for MLP across batch sizes.
    
    Args:
        dataset: MLPDataset to train on.
        optimizer_type: Either 'SGD' or 'Adam'.
        batch_sizes: List of batch sizes to test.
        num_iterations: Number of training iterations per run.
        lr_base: Base learning rate for sweep (uses default if None).
        verbose: Whether to print progress.
    
    Returns:
        ScalingLawResult containing sweep results and analysis.
    """
    # Adam typically needs smaller learning rates
    if lr_base is None:
        lr_base = 0.0001 if optimizer_type == "Adam" else LR_BASE
    
    learning_rates = generate_learning_rates(base=lr_base)
    results: dict[int, dict[float, float]] = {}
    
    for batch_size in batch_sizes:
        results[batch_size] = {}
        
        for lr in learning_rates:
            try:
                result = train_mlp(
                    dataset=dataset,
                    num_iterations=num_iterations,
                    batch_size=batch_size,
                    learning_rate=lr,
                    optimizer_type=optimizer_type,
                )
                final_loss = compute_final_loss(result.test_losses)
            except Exception:
                final_loss = DIVERGENCE_THRESHOLD
            
            results[batch_size][lr] = final_loss
        
        if verbose:
            print(f"Finished batch size {batch_size}")
    
    # Analyze results
    optimal_lrs, avg_optimal_lrs = find_optimal_learning_rates(results, batch_sizes)
    power_law_slope = fit_power_law(batch_sizes, avg_optimal_lrs)
    
    return ScalingLawResult(
        batch_sizes=batch_sizes,
        optimal_lrs=optimal_lrs,
        avg_optimal_lrs=avg_optimal_lrs,
        power_law_slope=power_law_slope,
        results=results,
    )

In [ ]:
###### TODO: Q2 - Run scaling law experiment for MLP with SGD

mlp_sgd_scaling_result = run_mlp_lr_sweep(
    dataset=mlp_dataset,
    optimizer_type="SGD",
)

# Plot results
plot_lr_sweep_results(
    mlp_sgd_scaling_result.results,
    mlp_sgd_scaling_result.batch_sizes,
    title="Learning Rate vs Test Loss (MLP with SGD)",
)

plot_scaling_law(
    mlp_sgd_scaling_result,
    title="Batch Size vs Optimal Learning Rate (MLP with SGD)",
)

print(f"\nMLP with SGD: The relationship follows a power law: LR ~ BS^{mlp_sgd_scaling_result.power_law_slope:.2f}")
print(f"Comparison with Least-Squares SGD (slope={linear_scaling_result.power_law_slope:.2f}):")
print("The MLP scaling law shows a similar trend - optimal learning rate increases with batch size.")

######

## Q3: MLP with Adam

Finally, we will repeat the scaling law curve, but using the Adam optimizer. This time, implement a learning rate schedule, such that the learning rate has a linear warmup for 10 steps, then uses cosine decay for the rest of training. Use a beta1=0.5, beta2=0.5, and a weight decay of 0.001. As before, plot the comparison curves, then plot a curve of the batch size vs. optimal learning rate. Does the scaling for learning rate change with Adam vs. SGD?

In [ ]:
##### TODO: Q3 - Run scaling law experiment for MLP with Adam

mlp_adam_scaling_result = run_mlp_lr_sweep(
    dataset=mlp_dataset,
    optimizer_type="Adam",
)

# Plot results
plot_lr_sweep_results(
    mlp_adam_scaling_result.results,
    mlp_adam_scaling_result.batch_sizes,
    title="Learning Rate vs Test Loss (MLP with Adam)",
)

plot_scaling_law(
    mlp_adam_scaling_result,
    title="Batch Size vs Optimal Learning Rate (MLP with Adam)",
)

######

In [ ]:
# Final comparison summary
print("=" * 50)
print("SCALING LAW COMPARISON SUMMARY")
print("=" * 50)
print(f"\nLeast-Squares SGD: LR ~ BS^{linear_scaling_result.power_law_slope:.2f}")
print(f"MLP with SGD:      LR ~ BS^{mlp_sgd_scaling_result.power_law_slope:.2f}")
print(f"MLP with Adam:     LR ~ BS^{mlp_adam_scaling_result.power_law_slope:.2f}")
print("\n" + "=" * 50)
print("OBSERVATIONS")
print("=" * 50)
print("""
- SGD methods typically show near-linear scaling (slope ~ 1.0)
- Adam typically shows weaker scaling with batch size
- Adam's adaptive learning rates help compensate for batch size changes
- The optimal learning rate for Adam may scale sub-linearly with batch size
""")